# Luggage Email/Outbound — EDA

Exploratory analysis of all luggage weight variant (L5B15/L5B20/L5B25/L5B30/L5B40/L5B50/CLUG)
Email/Outbound/Luggage/Sales scoring events.
Reads from `data/processed/luggage_email_outbound.parquet` — run `02_data_ingestion.ipynb` first.

In [ ]:
from pathlib import Path
from collections import Counter
import json as _json

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

print("Imports OK")

In [ ]:
DECISIONS_DIR = Path("../data/decisions")
PROCESSED_FILE = Path("../data/processed/luggage_email_outbound.parquet")
df_model = pd.read_parquet(PROCESSED_FILE)
print(f"Loaded {len(df_model):,} rows, {df_model.shape[1]} columns")
print("pyName breakdown:")
print(df_model["pyName"].value_counts().to_string())
df_model.head(3)

In [ ]:
import sys
sys.path.insert(0, "../src")
from my_project.features import VARIANT_FEATURES

_cfg             = VARIANT_FEATURES["L5B15"]
PEGA_FEATURES    = list(_cfg.features)   # local aliases keep downstream cells unchanged
NUMERIC_FEATURES = _cfg.numeric

active     = [f for f in PEGA_FEATURES if f in df_model.columns]
cat_active = [f for f in active if f not in NUMERIC_FEATURES]
num_active = [f for f in active if f     in NUMERIC_FEATURES]
print(f"Active predictors: {len(active)}  ({len(num_active)} numeric, {len(cat_active)} categorical)")


def _ns(col: str) -> str:
    if col.startswith("IH."):                           return "IH.*"
    if col.startswith("param::"):                       return "param::Param.*"
    if col.startswith("CustBookedFlight.BookingData."): return "CustBookedFlight.BookingData.*"
    if col.startswith("CustBookedFlight.FlightData."):  return "CustBookedFlight.FlightData.*"
    return col.split(".")[0]


NS_COLORS = {
    "IH.*":                           "#1f77b4",
    "param::Param.*":                 "#ff7f0e",
    "CustBookedFlight.BookingData.*": "#2ca02c",
    "CustBookedFlight.FlightData.*":  "#d62728",
}

## Temporal Distribution

When were these decisions made? Uses  — the actual Pega interaction timestamp.

In [ ]:
COLORS = {
    "CLUG":  "#1f77b4",
    "L5B15": "#ff7f0e",
    "L5B25": "#2ca02c",
    "L5B30": "#d62728",
    "L5B20": "#9467bd",
    "L5B40": "#8c564b",
    "L5B50": "#e377c2",
}

# Parse decision time if not already datetime
if df_model["pxDecisionTime"].dtype == "object":
    dt_col = pd.to_datetime(
        df_model["pxDecisionTime"], format="%Y%m%dT%H%M%S.%f %Z", utc=True, errors="coerce"
    )
else:
    dt_col = df_model["pxDecisionTime"]

df_t = df_model.copy()
df_t["week"] = dt_col.dt.to_period("W").dt.start_time

has_time = df_t["week"].notna().sum()
print(f"{has_time:,} of {len(df_t):,} rows have a valid pxDecisionTime")

pv = (
    df_t.dropna(subset=["week"])
    .groupby(["week", "pyName"]).size()
    .reset_index(name="count")
    .pivot(index="week", columns="pyName", values="count")
    .fillna(0)
)
col_order = pv.sum().sort_values(ascending=False).index.tolist()
pv = pv[col_order]

fig, ax = plt.subplots(figsize=(13, 4))
ax.stackplot(
    pv.index,
    [pv[c] for c in col_order],
    labels=col_order,
    colors=[COLORS.get(c, "#aaa") for c in col_order],
    alpha=0.82,
)
ax.set_title("Email/Outbound luggage decisions by interaction date (weekly)", fontsize=12, fontweight="bold")
ax.set_ylabel("Decisions / week")
ax.set_xlabel("Interaction date (pxDecisionTime)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.tick_params(axis="x", rotation=35)
ax.legend(loc="upper left", fontsize=9, framealpha=0.88, ncol=2)
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.text(0.99, 0.97, f"n = {has_time:,} decisions",
        transform=ax.transAxes, ha="right", va="top", fontsize=9, color="#333")

plt.tight_layout()
plt.savefig("../data/luggage_decisions_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

## Data Quality Assessment

Assess missingness, constant columns, target distribution, feature correlations, and categorical cardinality.

In [ ]:
### 6.2 Missingness and Constants
summary = pd.DataFrame({
    "dtype": df_model.dtypes.astype(str),
    "missing_frac": df_model.isna().mean(),
    "n_unique": df_model.nunique(dropna=True)
}).sort_values(["missing_frac", "n_unique"], ascending=[False, True])

summary.head(50)

In [ ]:
# constants check
constant_cols = summary[summary["n_unique"] <= 1]
constant_cols

In [ ]:
### 6.3 Propensity Distribution by Variant
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# All variants combined
axes[0].hist(df_model["propensity"].dropna(), bins=60, color="steelblue", alpha=0.8)
axes[0].set_xlabel("Propensity")
axes[0].set_ylabel("Count")
axes[0].set_title("All variants — propensity distribution")

# Per variant
for name, grp in df_model.groupby("pyName"):
    axes[1].hist(grp["propensity"].dropna(), bins=40, alpha=0.6, label=name)
axes[1].set_xlabel("Propensity")
axes[1].set_title("Per variant — propensity distribution")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("../data/propensity_distribution.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
### 6.4  Predictor correlation with propensity (active features, Spearman)
# Numeric features converted via pd.to_numeric; categoricals label-encoded.
# Spearman is used throughout — most active predictors are symbolic.

enc_prop = {}
for col in active:
    s = df_model[col]
    if col in NUMERIC_FEATURES:
        enc_prop[col] = pd.to_numeric(s, errors="coerce")
    else:
        nan_mask = s.isna()
        codes, _ = pd.factorize(s.fillna("__NaN__"))
        col_enc = pd.Series(codes.astype(float), index=s.index)
        col_enc[nan_mask] = np.nan
        enc_prop[col] = col_enc

enc_prop_df = pd.DataFrame(enc_prop)
enc_prop_df["propensity"] = df_model["propensity"].astype(float)

corr_with_target = (
    enc_prop_df.corr(method="spearman")["propensity"]
    .drop("propensity")
    .abs()
    .sort_values(ascending=False)
    .rename("abs_spearman_rho")
    .to_frame()
)
display(corr_with_target)

In [ ]:
### 6.5  Categorical predictor cardinality (active features only)
pd.DataFrame({
    "n_unique":    [df_model[f].nunique(dropna=True) for f in cat_active],
    "missing_pct": [df_model[f].isna().mean().round(3) for f in cat_active],
}, index=cat_active).sort_values("n_unique", ascending=False)

## Active Predictor Analysis (6.6–6.9)

Detailed breakdown of the active predictors from `PEGA_FEATURES`, using types declared in `NUMERIC_FEATURES`.

| Section | Content |
|---|---|
| 6.6 Missingness by namespace | Min / max / mean missing rate per namespace |
| 6.7 Categorical cardinality | Unique values per symbolic predictor |
| 6.8 Spearman correlation | Full predictor × predictor correlation matrix |
| 6.9 Evidence & performance | ADM model metadata distributions |

In [ ]:
### 6.6  Missingness profile by namespace
# Summarises min/max/mean missing fraction per namespace across active predictors.

import matplotlib.ticker as mticker

miss_df = pd.DataFrame({
    "feature":      active,
    "namespace":    [_ns(f) for f in active],
    "missing_frac": [df_model[f].isna().mean() for f in active],
})

ns_summary = (
    miss_df.groupby("namespace")["missing_frac"]
    .agg(n_features="count", miss_min="min", miss_max="max", miss_mean="mean")
    .sort_values("miss_mean", ascending=False)
    .reset_index()
)
display(
    ns_summary
    .assign(
        miss_min =ns_summary["miss_min"].map("{:.1%}".format),
        miss_max =ns_summary["miss_max"].map("{:.1%}".format),
        miss_mean=ns_summary["miss_mean"].map("{:.1%}".format),
    )
    .rename(columns={
        "namespace":  "Namespace",
        "n_features": "Features",
        "miss_min":   "Min missing",
        "miss_max":   "Max missing",
        "miss_mean":  "Mean missing",
    })
    .set_index("Namespace")
)

# Strip plot: one dot per active predictor, x = missing fraction, coloured by namespace
miss_sorted = miss_df.sort_values(["namespace", "missing_frac"], ascending=[True, False])
short_labels = (
    miss_sorted["feature"]
    .str.replace("CustBookedFlight.BookingData.", "BD.",  regex=False)
    .str.replace("CustBookedFlight.FlightData.",  "FD.",  regex=False)
    .str.replace("CustBookedFlight.",             "CBF.", regex=False)
    .str.replace("param::",                       "",     regex=False)
)

fig, ax = plt.subplots(figsize=(10, 8))
for ns, grp in miss_sorted.groupby("namespace"):
    mask = miss_sorted["namespace"] == ns
    ax.scatter(
        miss_sorted.loc[mask, "missing_frac"],
        short_labels[mask],
        color=NS_COLORS.get(ns, "#aaa"),
        label=ns, s=65, alpha=0.88,
    )
ax.axvline(0.5, color="tomato", linestyle="--", lw=1.2, alpha=0.75, label="50% threshold")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel("Missing fraction")
ax.set_title("Missingness per active predictor, by namespace", fontsize=12, fontweight="bold")
ax.legend(loc="lower right", fontsize=9, framealpha=0.88)
ax.grid(axis="x", alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()

In [ ]:
### 6.7  Cardinality of active categorical predictors

print(f"Categorical active predictors: {len(cat_active)}  |  Numeric: {len(num_active)}")

card = (
    pd.DataFrame({
        "feature":     cat_active,
        "namespace":   [_ns(f) for f in cat_active],
        "n_unique":    [df_model[f].nunique(dropna=True) for f in cat_active],
        "missing_pct": [df_model[f].isna().mean() for f in cat_active],
    })
    .sort_values("n_unique", ascending=False)
    .reset_index(drop=True)
    .assign(missing_pct=lambda d: d["missing_pct"].map("{:.1%}".format))
)
display(card.set_index("feature"))

# Derive short labels from card["feature"] AFTER sorting, not from cat_active
card["short"] = (
    card["feature"]
    .str.replace("CustBookedFlight.BookingData.", "BD.",  regex=False)
    .str.replace("CustBookedFlight.FlightData.",  "FD.",  regex=False)
    .str.replace("CustBookedFlight.",             "CBF.", regex=False)
    .str.replace("param::",                       "",     regex=False)
)

fig, ax = plt.subplots(figsize=(9, max(4, len(cat_active) * 0.42)))
bars = ax.barh(
    card["short"],
    card["n_unique"],
    color=[NS_COLORS.get(_ns(f), "#aaa") for f in card["feature"]],
    alpha=0.85,
)
ax.bar_label(bars, padding=3, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Unique non-null values")
ax.set_title(
    "Cardinality of active categorical predictors (PEGA_FEATURES only)",
    fontsize=12, fontweight="bold",
)
ax.grid(axis="x", alpha=0.3, linestyle="--")
handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.85) for c in NS_COLORS.values()]
ax.legend(handles, NS_COLORS.keys(), fontsize=8, loc="lower right", framealpha=0.88)
plt.tight_layout()
plt.show()

In [ ]:
### 6.8  Spearman correlation structure among active predictors
# Numeric features  → pd.to_numeric (handles string-stored numbers from Pega JSON).
# Categorical features → pd.factorize label-encoding (rank-stable, NaN preserved).
# Columns that are all-NaN or constant are excluded before correlation (they would
# produce NaN ρ values and appear white in the heatmap).

enc_dict = {}
for col in active:
    s = df_model[col]
    if col in NUMERIC_FEATURES:
        enc_dict[col] = pd.to_numeric(s, errors="coerce")
    else:
        nan_mask = s.isna()
        codes, _ = pd.factorize(s.fillna("__NaN__"))
        col_enc = pd.Series(codes.astype(float), index=s.index)
        col_enc[nan_mask] = np.nan
        enc_dict[col] = col_enc

enc = pd.DataFrame(enc_dict)

# Drop columns where correlation is undefined (all-NaN or zero variance)
valid_cols = [
    c for c in enc.columns
    if enc[c].notna().sum() > 1 and enc[c].nunique(dropna=True) > 1
]
excluded = [c for c in enc.columns if c not in valid_cols]
if excluded:
    print(f"Excluded from correlation matrix (all-NaN or constant): {excluded}")

enc_valid = enc[valid_cols]
spearman_corr = enc_valid.corr(method="spearman")

short_sp = [
    f.replace("CustBookedFlight.BookingData.", "BD.")
     .replace("CustBookedFlight.FlightData.",  "FD.")
     .replace("CustBookedFlight.",             "CBF.")
     .replace("IH.Email.Outbound.",            "IH.Eml.Out.")
     .replace("IH.Email.Inbound.",             "IH.Eml.In.")
     .replace("IH.Push.Outbound.",             "IH.Push.")
     .replace("IH.Event.Outbound.",            "IH.Evt.")
     .replace("param::Param.",                 "Param.")
    for f in valid_cols
]

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(spearman_corr.values, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.8, label="Spearman ρ")
ax.set_xticks(range(len(short_sp)))
ax.set_yticks(range(len(short_sp)))
ax.set_xticklabels(short_sp, rotation=45, ha="right", fontsize=7.5)
ax.set_yticklabels(short_sp, fontsize=7.5)
ax.set_title(
    "Spearman correlation matrix — active predictors\n"
    "(categoricals label-encoded; all-NaN / constant columns excluded)",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
plt.show()

# Tabulate the strongest pairwise correlations for narrative support
pairs = (
    spearman_corr
    .where(np.triu(np.ones(spearman_corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b", 0: "rho"})
    .assign(abs_rho=lambda d: d["rho"].abs())
    .sort_values("abs_rho", ascending=False)
)
print("Top correlated pairs (|ρ| > 0.30):")
display(
    pairs[pairs["abs_rho"] > 0.30][["feature_a", "feature_b", "rho"]]
    .head(20)
    .reset_index(drop=True)
)

In [ ]:
### 6.9  Model evidence and performance distributions
# modelEvidence: number of positive/negative outcomes the ADM model has seen
# modelPerformance (AUC): how well the model discriminates at the time of scoring
# Both vary per row because Pega re-snapshots the model on every scoring event.

for col in ["modelEvidence", "modelPerformance"]:
    s = df_model[col].dropna()
    print(f"\n── {col}  (n={len(s):,}, {df_model[col].isna().mean():.1%} missing)")
    print(s.describe(percentiles=[0.25, 0.5, 0.75]).to_frame().T.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, color in zip(
    axes,
    ["modelEvidence", "modelPerformance"],
    ["steelblue",     "seagreen"],
):
    s = df_model[col].dropna()
    ax.hist(s, bins=60, color=color, alpha=0.82, edgecolor="white")
    ax.axvline(s.median(), color="tomato", linestyle="--", lw=1.5,
               label=f"median = {s.median():.4f}")
    ax.axvline(s.mean(),   color="gold",   linestyle=":",  lw=1.5,
               label=f"mean = {s.mean():.4f}")
    ax.set_title(col, fontsize=12, fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Decisions")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3, linestyle="--")

fig.suptitle(
    "ADM model metadata — evidence and AUC across all scoring events",
    fontsize=11, fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
### Publication figure — model performance (AUC) distribution

import matplotlib as mpl
from matplotlib.ticker import FuncFormatter

PUB_RC = {
    "font.family":        "sans-serif",
    "font.size":          9,
    "axes.linewidth":     0.7,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "xtick.major.size":   3.5,
    "ytick.major.size":   3.5,
    "xtick.major.width":  0.7,
    "ytick.major.width":  0.7,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "legend.frameon":     False,
    "legend.fontsize":    8,
}

with mpl.rc_context(PUB_RC):
    fig, ax = plt.subplots(figsize=(3.5, 2.6))

    s = df_model["modelPerformance"].dropna()
    med, mn = s.median(), s.mean()

    ax.hist(s, bins=55, color="#2A7A5A", alpha=0.80,
            edgecolor="white", linewidth=0.3, zorder=2)
    ax.axvline(med, color="#C0392B", linestyle="--",
               linewidth=1.1, label=f"Median = {med:.4f}", zorder=3)
    ax.axvline(mn,  color="#E67E22", linestyle=":",
               linewidth=1.1, label=f"Mean = {mn:.4f}",   zorder=3)

    ax.set_xlabel("Model performance (AUC)", labelpad=5)
    ax.set_ylabel("Frequency")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.grid(axis="y", linewidth=0.35, color="#DDDDDD", zorder=0)
    ax.set_axisbelow(True)
    ax.legend(handlelength=1.8, handletextpad=0.5)

    fig.tight_layout(pad=1.0)

    out = Path("../data/fig_model_performance")
    fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out.with_suffix(".png"), dpi=300, bbox_inches="tight")
    print(f"Saved → {out}.pdf / .png")
    plt.show()

### 6.10  Implications: filtering to the production-decision model

The bimodal AUC distribution in §6.9 is not a learning curve — it is the signature of **two Pega ADM techniques scoring each event in parallel**. The `modelTechnique` column has two values, `"NaiveBayes"` and a degenerate `"0.0"` label (the technique name failed to serialise to a string in our exports). Every variant carries a near-50/50 split between the two, with disjoint `modelVersion` UUID sets across the techniques — confirming two distinct models score each event in parallel rather than alternating over time.

**Which model drives decisions:** the `"0.0"`-labelled model is the production-decision model — i.e., the one whose propensity actually selects the customer-facing offer (confirmed by the Transavia data team). The parallel NaiveBayes scorer runs as an audit/shadow comparator and does not drive offer selection.

**Implications for the surrogate analysis (notebooks 04–08):**

- A CatBoost surrogate trained on the *mixed* dataset is approximating two different scoring functions $f_{prod}, f_{NB}$ with one model — a mixture-target problem that bounds achievable fidelity regardless of model capacity.
- SHAP / LIME / DT explanations of such a mixture surrogate would explain the *average* behaviour of the two techniques, not the one that drives real decisions.
- Stability metrics across temporal/route splits can be confounded if the two techniques are unevenly distributed across the splits.

**Decision:** all downstream notebooks (`04_surrogate_selection`, `05_surrogate_fit`, `06_explanation`, `07_stability`, `08_attribution`) restrict to `modelTechnique == "0.0"` — the production-decision model. The NaiveBayes scorer is excluded from primary analysis (its scores carry online-learning state invisible to the surrogate, capping fidelity at R²~0.4; the production model is feature-deterministic and gives R²~0.9, see preliminary fits below). Sample sizes after filtering: L5B15 31,709 / CLUG 16,984 / BookingDotCom 19,866 / Cartrawler 20,645 — all comfortably sufficient.


In [ ]:
# Per-variant × technique breakdown — confirms the parallel-technique pattern.
# Near-50/50 split per variant with distinct modelVersion UUIDs across techniques
# rules out the "alternating over time" interpretation.
tech_breakdown = (
    df_model.groupby(["pyName", "modelTechnique"], dropna=False)
    .agg(
        rows=("modelPerformance", "size"),
        mean_AUC=("modelPerformance", "mean"),
        unique_versions=("modelVersion", "nunique"),
    )
    .round({"mean_AUC": 4})
)
display(
    tech_breakdown.style
    .format({"rows": "{:,}", "mean_AUC": "{:.4f}", "unique_versions": "{:,}"})
    .background_gradient(subset=["mean_AUC"], cmap="RdYlGn", vmin=0.5, vmax=0.85)
    .set_caption("Per-variant × modelTechnique: row counts, mean AUC, and number of distinct modelVersion UUIDs. Near-equal row counts within each variant + disjoint UUID sets confirm two techniques running in parallel.")
)


In [ ]:
### 6.10b  Publication figure — production-model AUC distribution only
# Mirrors the §6.9 publication figure but filters to modelTechnique == "0.0"
# (the production-decision model). Removes the lower-AUC NaiveBayes peak,
# showing the unimodal distribution of the surrogate's actual target model.

import matplotlib as mpl
from matplotlib.ticker import FuncFormatter

PUB_RC = {
    "font.family":        "sans-serif",
    "font.size":          9,
    "axes.linewidth":     0.7,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "xtick.major.size":   3.5,
    "ytick.major.size":   3.5,
    "xtick.major.width":  0.7,
    "ytick.major.width":  0.7,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
    "legend.frameon":     False,
    "legend.fontsize":    8,
}

with mpl.rc_context(PUB_RC):
    fig, ax = plt.subplots(figsize=(3.5, 2.6))

    s = df_model.loc[df_model["modelTechnique"] == "0.0", "modelPerformance"].dropna()
    med, mn = s.median(), s.mean()

    ax.hist(s, bins=55, color="#2A7A5A", alpha=0.80,
            edgecolor="white", linewidth=0.3, zorder=2)
    ax.axvline(med, color="#C0392B", linestyle="--",
               linewidth=1.1, label=f"Median = {med:.4f}", zorder=3)
    ax.axvline(mn,  color="#E67E22", linestyle=":",
               linewidth=1.1, label=f"Mean = {mn:.4f}",   zorder=3)

    ax.set_xlabel("Model performance (AUC) — production model only", labelpad=5)
    ax.set_ylabel("Frequency")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.grid(axis="y", linewidth=0.35, color="#DDDDDD", zorder=0)
    ax.set_axisbelow(True)
    ax.legend(handlelength=1.8, handletextpad=0.5)

    fig.tight_layout(pad=1.0)

    out = Path("../data/fig_model_performance_prod")
    fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(out.with_suffix(".png"), dpi=300, bbox_inches="tight")
    print(f"Saved → {out}.pdf / .png")
    print(f"n = {len(s):,}  |  range = [{s.min():.4f}, {s.max():.4f}]")
    plt.show()


## Offering Breakdown

Which Sales offerings appear most in this export, and do actual purchase outcomes exist?

## Channel Overview

How many decisions and interactions exist per channel, and which offerings dominate each one?

In [ ]:
CH_COLORS = {'Email': '#1f77b4', 'Mobile': '#2ca02c', 'Web': '#ff7f0e', 'Push': '#9467bd'}
GROUP_CMAP = plt.cm.tab20

rows_ch = []
for filepath in sorted(DECISIONS_DIR.glob("*.json")):
    with open(filepath, encoding="utf-8") as _f:
        for line in _f:
            line = line.strip()
            if not line: continue
            rec = _json.loads(line)
            for dr in rec.get("pxDecisionResults", []):
                rows_ch.append({
                    "pxInteractionID": rec.get("pxInteractionID"),
                    "pyChannel":   dr.get("pyChannel"),
                    "pyDirection": dr.get("pyDirection"),
                    "pyGroup":     dr.get("pyGroup"),
                    "pyIssue":     dr.get("pyIssue"),
                    "pyName":      dr.get("pyName"),
                })

df_ch = pd.DataFrame(rows_ch)

print(f"Total unique interactions: {df_ch['pxInteractionID'].nunique():,}")
print()
print("Decisions per channel / direction:")
print(df_ch.groupby(["pyChannel","pyDirection"]).size()
           .reset_index(name="decisions")
           .sort_values("decisions", ascending=False)
           .to_string(index=False))

fig = plt.figure(figsize=(15, 14))
fig.suptitle("Decision data overview — all channels", fontsize=14, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.52, wspace=0.38)

# Total decisions per channel
ax1 = fig.add_subplot(gs[0, 0])
ch_tot = df_ch.groupby("pyChannel").size().sort_values(ascending=False)
bars = ax1.bar(ch_tot.index, ch_tot.values,
               color=[CH_COLORS.get(c,"#aaa") for c in ch_tot.index], alpha=0.87)
ax1.bar_label(bars, labels=[f"{v:,}" for v in ch_tot.values], padding=3, fontsize=9)
ax1.set_title("Total decisions per channel", fontsize=11, fontweight="bold")
ax1.set_ylabel("Decisions")
ax1.grid(axis="y", alpha=0.3, linestyle="--")
ax1.set_ylim(0, ch_tot.max() * 1.15)

# Unique interactions per channel
ax2 = fig.add_subplot(gs[0, 1])
int_per_ch = df_ch.groupby("pyChannel")["pxInteractionID"].nunique().sort_values(ascending=False)
bars2 = ax2.bar(int_per_ch.index, int_per_ch.values,
                color=[CH_COLORS.get(c,"#aaa") for c in int_per_ch.index], alpha=0.87)
ax2.bar_label(bars2, labels=[f"{v:,}" for v in int_per_ch.values], padding=3, fontsize=9)
ax2.set_title("Unique interactions per channel", fontsize=11, fontweight="bold")
ax2.set_ylabel("Interactions (pxInteractionID)")
ax2.grid(axis="y", alpha=0.3, linestyle="--")
ax2.set_ylim(0, int_per_ch.max() * 1.15)

# Top offerings per channel
for ch, pos in zip(["Email","Mobile","Web","Push"], [gs[1,0],gs[1,1],gs[2,0],gs[2,1]]):
    ax = fig.add_subplot(pos)
    sub = (df_ch[df_ch["pyChannel"] == ch]
           .groupby(["pyName","pyGroup"]).size()
           .reset_index(name="n")
           .sort_values("n", ascending=False)
           .head(12))
    groups = sub["pyGroup"].unique()
    grp_color = {g: GROUP_CMAP(i / max(len(groups)-1, 1)) for i, g in enumerate(groups)}
    bars = ax.barh(sub["pyName"], sub["n"],
                   color=[grp_color[g] for g in sub["pyGroup"]], alpha=0.85)
    ax.bar_label(bars, labels=[f"{v:,}" for v in sub["n"]], padding=3, fontsize=7.5)
    ax.invert_yaxis()
    ax.set_title(f"{ch}  —  {df_ch[df_ch['pyChannel']==ch].shape[0]:,} decisions total",
                 fontsize=10, fontweight="bold", color=CH_COLORS.get(ch,"#333"))
    ax.set_xlabel("Decisions", fontsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="x", alpha=0.3, linestyle="--")
    ax.set_xlim(0, sub["n"].max() * 1.18)
    handles = [plt.Rectangle((0,0),1,1, color=grp_color[g], alpha=0.85) for g in groups]
    ax.legend(handles, groups, fontsize=6.5, loc="lower right", framealpha=0.8)

plt.tight_layout()
plt.savefig("../data/channel_overview.png", dpi=140, bbox_inches="tight")
plt.show()

### Key observations

**Scale**
The export contains **16,184 unique customer interactions** resulting in **63,897 individual decisions** — meaning Pega scores roughly 4 offerings per interaction on average, though this varies significantly by channel.

**Email (43,049 decisions — 6,205 interactions)**
The largest channel by far, with ~7 offerings scored per interaction. The content mix is broad: FAQs and informational items (TravelDocs, OnlineCheckin) dominate the top, followed by ThirdParty cross-sells (Cartrawler, GetYourGuide, Booking.com). Luggage add-ons are present but are a smaller slice of the total — the email channel is clearly used for a wide range of customer communications beyond ancillary sales.

**Mobile (11,102 decisions — 5,551 interactions)**
Almost exclusively Luggage. CLUG and L5B15 account for the vast majority of decisions, with other ancillaries (FISH, BIKE, seats) making up a small tail. With ~2 offerings per interaction, Mobile is the most focused channel — customers in-app are shown a tight luggage upsell.

**Web (7,639 decisions — 2,579 interactions)**
Similar profile to Mobile but with a slightly broader mix — Luggage still leads (CLUG, L5B15, L5B25) alongside ThirdParty offers like Cartrawler and GetYourGuide. ~3 offerings per interaction.

**Push (2,107 decisions — 2,107 interactions)**
The smallest channel and the most targeted: exactly 1 decision per interaction, confirming Push is a single-nudge mechanism. Content is almost entirely Seats (SeatFirstRow, SeatUpfront, SeatExtraLegroom), making Push a dedicated seat upgrade channel.

---
> **Note on outcomes:** `pyOutcome` is blank for all records in this export. The data captures Pega's *scoring decisions* (what was offered and at what propensity), not whether the customer actually purchased. Ground-truth conversion labels would need to be joined in separately from a booking/transaction system.

### ThirdParty offerings

## Cross-variant Missingness Profile
Feature missingness summarised by group across all four analysed variants. Numeric features are checked for `NaN`; categorical features for `__MISSING__`. Both are included in the mean missingness percentage.

In [ ]:
# ── Cross-variant missingness table ──────────────────────────────────────
import sys
from my_project.features import VARIANT_FEATURES
from my_project.surrogate import build_feature_matrix

_lug = pd.read_parquet(Path("../data/processed/luggage_email_outbound.parquet"))
_tp  = pd.read_parquet(Path("../data/processed/thirdparty_email_outbound.parquet"))

_VARIANTS_DATA = {
    "L5B15":         _lug[_lug["pyName"] == "L5B15"].reset_index(drop=True),
    "CLUG":          _lug[_lug["pyName"] == "CLUG"].reset_index(drop=True),
    "BookingDotCom": _tp[_tp["pyName"]   == "BookingDotCom"].reset_index(drop=True),
    "Cartrawler":    _tp[_tp["pyName"]   == "Cartrawler"].reset_index(drop=True),
}

def _feature_group(name):
    if name.startswith("param::"):                      return "Scoring parameters"
    if name.startswith("IH."):                          return "Contact history"
    if name.startswith("Customer."):                    return "Customer attributes"
    if name.startswith("CustBookedFlight.FlightData."): return "Flight data"
    return "Booking context"

def _miss_pct(X, cat_cols):
    result = {}
    for col in X.columns:
        if col in cat_cols:
            result[col] = (X[col] == "__MISSING__").sum() / len(X) * 100
        else:
            result[col] = X[col].isna().sum() / len(X) * 100
    return pd.Series(result)

GROUPS = ["Booking context", "Flight data", "Contact history", "Scoring parameters", "Customer attributes"]
_results = {}
for variant, df in _VARIANTS_DATA.items():
    cfg = VARIANT_FEATURES[variant]
    X, y, cat_cols, num_cols = build_feature_matrix(df, list(cfg.features), cfg.numeric)
    mp = _miss_pct(X, cat_cols)
    group_stats = {}
    for g in GROUPS:
        feats = [f for f in X.columns if _feature_group(f) == g]
        if feats:
            group_stats[g] = {"n": len(feats), "mean": round(mp[feats].mean(), 1)}
        else:
            group_stats[g] = {"n": 0, "mean": None}
    _results[variant] = group_stats

_variants = list(_VARIANTS_DATA.keys())
rows = []
for g in GROUPS:
    row = {"Feature group": g}
    for v in _variants:
        row[f"n ({v})"]    = _results[v][g]["n"] if _results[v][g]["n"] > 0 else "—"
        row[f"miss ({v})"] = f"{_results[v][g]['mean']:.1f}%" if _results[v][g]["mean"] is not None else "—"
    rows.append(row)

miss_df = pd.DataFrame(rows).set_index("Feature group")

print("Cross-variant missingness profile (mean % missing per feature group):")
print()
display(
    miss_df.style
    .set_caption("Feature groups and missingness across all analysed variants")
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
)

## Third-Party Variants — BookingDotCom & Cartrawler
EDA for the third-party cross-sell variants loaded from `data/processed/thirdparty_email_outbound.parquet`. Covers temporal distribution, propensity distribution, model version churn, and feature missingness. Mirrors the luggage EDA sections above.

In [ ]:
# ── Reload saved parquet (works even if parse cells were skipped) ─────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

df_tp = pd.read_parquet(Path("../data/processed/thirdparty_email_outbound.parquet"))
df_tp["pxDecisionTime"] = pd.to_datetime(df_tp["pxDecisionTime"], utc=True, errors="coerce")
VARIANTS = ["BookingDotCom", "Cartrawler"]
colors   = {"BookingDotCom": "#4c72b0", "Cartrawler": "#dd8452"}

print(f"Loaded {len(df_tp):,} rows")
print(df_tp["pyName"].value_counts().to_string())

In [ ]:
# ── Temporal distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=False)
for ax, variant in zip(axes, VARIANTS):
    sub = df_tp[df_tp["pyName"] == variant].copy()
    sub["week"] = sub["pxDecisionTime"].dt.to_period("W").dt.to_timestamp()
    weekly = sub.groupby("week").size()
    ax.bar(weekly.index, weekly.values, width=6, color=colors[variant], alpha=0.85)
    ax.set_title(f"{variant} — decisions per week", fontsize=10)
    ax.set_xlabel("Week")
    ax.set_ylabel("Scoring events")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()

for variant in VARIANTS:
    sub = df_tp[df_tp["pyName"] == variant]
    dt  = sub["pxDecisionTime"].dropna()
    print(f"{variant:20s}  n={len(sub):,}  {dt.min().date()} -> {dt.max().date()}")

In [ ]:
# ── Propensity score distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, variant in zip(axes, VARIANTS):
    sub = df_tp[df_tp["pyName"] == variant]["propensity"].dropna()
    ax.hist(sub, bins=50, color=colors[variant], alpha=0.85, edgecolor="white", linewidth=0.3)
    ax.set_title(f"{variant} — propensity distribution", fontsize=10)
    ax.set_xlabel("Propensity score")
    ax.set_ylabel("Count")
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.text(0.97, 0.95, f"mean={sub.mean():.3f} std={sub.std():.3f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))
plt.tight_layout()
plt.show()

In [ ]:
# ── Model version churn ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, variant in zip(axes, VARIANTS):
    sub = df_tp[df_tp["pyName"] == variant].copy()
    sub["week"] = sub["pxDecisionTime"].dt.to_period("W").dt.to_timestamp()
    perf = sub.groupby("week")["modelPerformance"].mean()
    ax.plot(perf.index, perf.values, color=colors[variant], marker="o", markersize=4)
    ax.set_title(f"{variant} — mean AUC per week", fontsize=10)
    ax.set_xlabel("Week")
    ax.set_ylabel("Mean modelPerformance (AUC)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.tick_params(axis="x", rotation=30)
    ax.grid(alpha=0.3, linestyle="--")
    n_versions = sub["modelVersion"].nunique()
    ax.text(0.03, 0.95, f"{n_versions} unique model versions",
            transform=ax.transAxes, va="top", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature missingness ───────────────────────────────────────────────────
from my_project.features import VARIANT_FEATURES
from my_project.surrogate import build_feature_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, variant in zip(axes, VARIANTS):
    if variant not in VARIANT_FEATURES:
        ax.set_title(f"{variant} — no feature config found")
        continue
    sub = df_tp[df_tp["pyName"] == variant].reset_index(drop=True)
    cfg = VARIANT_FEATURES[variant]
    X, y, cat_cols, num_cols = build_feature_matrix(sub, list(cfg.features), cfg.numeric)
    miss = (X == "__MISSING__").sum() / len(X) * 100
    miss = miss[miss > 0].sort_values(ascending=True)
    if len(miss) == 0:
        ax.set_title(f"{variant} — no missing values")
        continue
    ax.barh(miss.index, miss.values, color=colors[variant], alpha=0.85)
    ax.set_title(f"{variant} — feature missingness (%)", fontsize=10)
    ax.set_xlabel("% missing")
    ax.grid(axis="x", alpha=0.3, linestyle="--")
    ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.show()